# AblationWriting JMQ prompt review
Upload `jmq_prompt_review.parquet` when prompted. Parquet preserves multiline text and types better than CSV.

In [ ]:
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))
!pip -q install pyarrow
import pandas as pd, json
df = pd.read_parquet(path) if path.endswith('.parquet') else pd.read_csv(path)
print(df.shape)
display(df[['model_name','prompt_id','jmq_direct_winner','both_refused_excluded','test_r_l2_1','test_mmd_recovery']].head())


In [ ]:
from IPython.display import display, Markdown
def show_row(index):
    r = df.iloc[index]
    display(Markdown(f"## {r.model_name} · prompt {r.prompt_id}\n**JMQ winner:** {r.jmq_direct_winner}  \n**Both refused/excluded:** {r.both_refused_excluded}"))
    for label, column in [('Prompt','prompt'),('Human','human_output'),('Baseline','baseline_output'),('Ablated','ablated_output'),('Judge rationale','jmq_direct_raw')]:
        display(Markdown(f"### {label}\n```text\n{r[column]}\n```"))
    display(json.loads(r.model_stats_json))
show_row(0)


In [ ]:
# Examples: filter to significant direct losses, or inspect only non-refusal pairs.
clean = df[~df.both_refused_excluded]
display(clean.groupby(['model_name','jmq_direct_winner']).size().unstack(fill_value=0))
